In [ ]:
import os
foldername = '/home/edavenport/analysis/vel-assim-manuscript/forecasts/results_figs/skill/'
os.makedirs(foldername, exist_ok=True)

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
import pandas as pd
import numpy as np
from forecasts.forecast_utils import get_forecast_params

# ═══════════════════════════════════════════════════════════════════
#  FORECAST PARAMETERS — only change start_date to switch forecasts
#  Supported range: Sep 2012 – Jul 2013 (1st of month)
# ═══════════════════════════════════════════════════════════════════
start_date = pd.Timestamp('2013-01-01')

# TPOSE-Vel state estimate directory (varies by run; set manually for each forecast)
vel_estimate_data_dir = '/data/SO3/edavenport/tpose6/jan2013/run_iter14/'

p = get_forecast_params(start_date)

# ── Unpack for use in subsequent cells ───────────────────────────
start_date  = p.start_date
end_date    = p.end_date
month_str   = p.month_str
day_str     = p.day_str
year_str    = p.year_str

noTAO_data_dir          = p.noTAO_data_dir
noTAO_forecast_data_dir = p.noTAO_forecast_data_dir
vel_forecast_data_dir   = p.vel_forecast_data_dir
grid_dir                = p.grid_dir

ref_date        = p.ref_date
itPerFile       = p.itPerFile
delta_t         = p.delta_t
num_diags       = p.num_diags
intervals       = p.intervals
n_forecast_days = p.n_forecast_days
n_eval          = p.n_eval
eval_slice      = p.eval_slice
eval_start_date = p.eval_start_date
days            = p.days
eval_dates      = p.eval_dates
eval_months     = p.eval_months
month_bounds    = p.month_bounds
month_centers   = p.month_centers

print(f'Forecast     : {start_date.date()} → {end_date.date()} ({n_forecast_days} days)')
print(f'Eval months  : {eval_months}')
print(f'itPerFile    : {itPerFile}  |  delta_t: {delta_t:.0f} s')
print(f'noTAO est dir: {noTAO_data_dir}')
print(f'noTAO fct dir: {noTAO_forecast_data_dir}')
print(f'Vel est dir  : {vel_estimate_data_dir}')
print(f'Vel fct dir  : {vel_forecast_data_dir}')

### Load TPOSE Estimates and Forecasts

In [ ]:
import matplotlib.pyplot as plt
import cmocean.cm as cmo
import xarray as xr
from xmitgcm import open_mdsdataset
plt.rcParams['font.size'] = 14

# SST comes from diag_state (THETA), surface level
prefix = ['diag_state']

def open_tpose(data_dir):
    ds = open_mdsdataset(
        data_dir=data_dir, grid_dir=grid_dir,
        iters=intervals, prefix=prefix, ref_date=ref_date, delta_t=delta_t)
    for coord in ['XC', 'YC', 'Z', 'XG', 'YG']:
        ds[coord] = ds[coord].astype(float)
    return ds

### TPOSE-noVel State Estimate
ds_tpose_noTAO = open_tpose(noTAO_data_dir)

### TPOSE-noVel Forecast
ds_tpose_noTAO_forecast = open_tpose(noTAO_forecast_data_dir)

### TPOSE-Vel State Estimate
ds_tpose_vel = open_tpose(vel_estimate_data_dir)

### TPOSE-Vel Forecast
ds_tpose_vel_forecast = open_tpose(vel_forecast_data_dir)

### Load HYCOM T and OISST climatology

Note: GLORYS does not contain temperature (variables are `zos`, `uo`, `vo` only) and is therefore omitted from SST comparisons.

OISST monthly climatology (1993–2012) provides both the SST reference (`sst_mean`) and the variability baseline (`sst_std`).

In [ ]:
import os as _os

_HYCOM_DIR = '/data/SO3/edavenport/tpose6/hycom_data/'

def load_hycom_T_daily(start_date, end_date):
    """Load HYCOM temperature for a date window, returning daily-mean data.

    Reads hycom_T_{YYYY}_{MM:02d}.nc files, sub-daily steps averaged to daily.
    Returns None if no files exist for the requested window.
    """
    start_date = pd.Timestamp(start_date)
    end_date   = pd.Timestamp(end_date)

    months = pd.date_range(
        start=start_date.to_period('M').to_timestamp(),
        end=end_date.to_period('M').to_timestamp(),
        freq='MS',
    )
    files = [
        _HYCOM_DIR + f'hycom_T_{m.year}_{m.month:02d}.nc'
        for m in months
        if _os.path.exists(_HYCOM_DIR + f'hycom_T_{m.year}_{m.month:02d}.nc')
    ]

    if not files:
        return None

    ds = xr.open_mfdataset(files, combine='by_coords')
    ds = ds.sel(
        time=slice(start_date.strftime('%Y-%m-%d'),
                   end_date.strftime('%Y-%m-%d'))
    )
    ds = ds.resample(time='1D').mean()

    full_index = pd.date_range(start_date, end_date, freq='1D')
    ds = ds.reindex(time=full_index)

    return ds


hycom_T = load_hycom_T_daily(start_date, end_date)
oisst   = xr.open_dataset('forecasts/oisst_data/oisst_climatology_1993to2012.nc')

if hycom_T is None:
    print('WARNING: No HYCOM T data available for this forecast window — HYCOM will be omitted from figures.')
else:
    print('HYCOM T loaded. Variables:', list(hycom_T.data_vars))

print('OISST climatology loaded. Variables:', list(oisst.data_vars))

### Compare Skill

In [ ]:
lonMin = 180
lonMax = 260
latMin = -10
latMax = 10

### SST RMSE vs OISST

In [ ]:
# ── Load TPOSE THETA at surface (Z=0) ────────────────────────────────────────
def load_theta_surface(ds):
    return (
        ds.THETA
          .isel(Z=0)
          .sel(XC=slice(lonMin, lonMax), YC=slice(latMin, latMax))
          .isel(time=eval_slice)
          .compute()
    )

tpose_noTAO_eval     = load_theta_surface(ds_tpose_noTAO)
tpose_noTAO_fct_eval = load_theta_surface(ds_tpose_noTAO_forecast)
tpose_vel_est_eval   = load_theta_surface(ds_tpose_vel)
tpose_vel_eval       = load_theta_surface(ds_tpose_vel_forecast)

tpose_xc = tpose_noTAO_eval.XC.values
tpose_yc = tpose_noTAO_eval.YC.values

# ── OISST reference: convert to 0:360, interpolate monthly mean to TPOSE grid ─
oisst_conv = oisst.assign_coords(longitude=(oisst.longitude % 360)).sortby('longitude')
oisst_clim_region = oisst_conv.sst_mean.sel(
    latitude=slice(latMin, latMax), longitude=slice(lonMin, lonMax)
)

# Build daily OISST reference by mapping each eval day to its calendar month
oisst_daily = np.stack([
    oisst_clim_region
      .sel(month=d.month)
      .interp(longitude=tpose_xc, latitude=tpose_yc, method='linear')
      .values
    for d in eval_dates
])  # (n_eval, n_lat, n_lon)

# ── HYCOM T surface extraction — skipped if no data for this window ───────────
if hycom_T is not None:
    hycom_T_conv = hycom_T.assign_coords(lon=(hycom_T.lon % 360)).sortby('lon')
    hycom_T_surf = (
        hycom_T_conv.water_temp
          .sel(depth=0.0, lat=slice(latMin, latMax), lon=slice(lonMin, lonMax))
          .isel(time=eval_slice)
          .interp(lon=tpose_xc, lat=tpose_yc, method='linear')
          .compute()
    ).values  # (n_eval, n_lat, n_lon)
    print('HYCOM T surface extracted. Shape:', hycom_T_surf.shape)

print('TPOSE THETA surface extracted. Shape:', tpose_noTAO_eval.shape)

In [ ]:
# ── Latitude-weighted spatial RMSE ────────────────────────────────────────────
lat_weights = np.cos(np.deg2rad(tpose_yc))
lat_weights_2d   = lat_weights[:, np.newaxis] * np.ones(len(tpose_xc))
lat_weights_norm = lat_weights_2d / np.nansum(lat_weights_2d)

def spatial_rmse(model_vals, obs_vals):
    """Weighted spatial RMSE; shape (n_time, n_lat, n_lon) → (n_time,)."""
    diff = model_vals - obs_vals
    return np.sqrt(np.nansum(diff**2 * lat_weights_norm, axis=(1, 2)))

rmse_noTAO          = spatial_rmse(tpose_noTAO_eval.values,     oisst_daily)
rmse_noTAO_forecast = spatial_rmse(tpose_noTAO_fct_eval.values, oisst_daily)
rmse_vel_estimate   = spatial_rmse(tpose_vel_est_eval.values,   oisst_daily)
rmse_vel            = spatial_rmse(tpose_vel_eval.values,        oisst_daily)

if hycom_T is not None:
    rmse_hycom = spatial_rmse(hycom_T_surf, oisst_daily)

# Persistence: initial field (day 1) held constant for all n_eval days
def persistence_rmse(arr):
    field = arr[0]
    return spatial_rmse(np.broadcast_to(field, (n_eval,) + field.shape), oisst_daily)

rmse_noTAO_fct_persistence = persistence_rmse(tpose_noTAO_fct_eval.values)
rmse_vel_persistence       = persistence_rmse(tpose_vel_eval.values)

# ── OISST std dev baseline: area-weighted mean of sst_std per calendar month ──
oisst_std_region = oisst_conv.sst_std.sel(
    latitude=slice(latMin, latMax), longitude=slice(lonMin, lonMax)
)
clim_lat_w      = np.cos(np.deg2rad(oisst_std_region.latitude.values))
clim_lat_w_2d   = clim_lat_w[:, np.newaxis] * np.ones(len(oisst_std_region.longitude))
clim_lat_w_norm = clim_lat_w_2d / np.nansum(clim_lat_w_2d)

month_baseline = {
    m: float(np.nansum(oisst_std_region.sel(month=m).values * clim_lat_w_norm))
    for m in eval_months
}

eval_dates_ts = pd.date_range(eval_start_date, periods=n_eval)
baseline_ts   = np.array([month_baseline[d.month] for d in eval_dates_ts])

for m in eval_months:
    print(f'Month {m} SST std dev baseline: {month_baseline[m]:.3f} °C')

In [ ]:
import seaborn as sns
import matplotlib.gridspec as gridspec

ylim = (0, 3.0)

# Each entry: (data, long_label, short_label, color, linestyle)
datasets_all = [
    (rmse_noTAO,                'TPOSE-noVel State Estimate', 'noVel Est',  'C0', '-'),
    (rmse_noTAO_forecast,       'TPOSE-noVel Forecast',       'noVel Fct',  'C0', '--'),
    (rmse_noTAO_fct_persistence,'TPOSE-noVel Persistence',    'noVel Pers', 'C0', ':'),
    (rmse_vel_estimate,         'TPOSE-Vel State Estimate',   'Vel Est',    'C1', '-'),
    (rmse_vel,                  'TPOSE-Vel Forecast',         'Vel Fct',    'C1', '--'),
    (rmse_vel_persistence,      'TPOSE-Vel Persistence',      'Vel Pers',   'C1', ':'),
]
if hycom_T is not None:
    datasets_all.append((rmse_hycom, 'HYCOM', 'HYCOM', 'C3', '-'))

datasets_tpose = datasets_all[:6]

for datasets, suffix in [(datasets_all, ''), (datasets_tpose, '_tpose_only')]:
    short_names = [ds[2] for ds in datasets]
    palette     = [ds[3] for ds in datasets]

    fig = plt.figure(figsize=(14, 5))
    gs  = gridspec.GridSpec(1, 2, width_ratios=[4, 1], wspace=0.15)
    ax  = fig.add_subplot(gs[0])
    ax_b = fig.add_subplot(gs[1])

    # ── Time series ───────────────────────────────────────────────
    for data, long_name, _, color, ls in datasets:
        ax.plot(days, data, color=color, lw=2, ls=ls, label=long_name)
    ax.plot(days, baseline_ts, color='k', lw=1.5, ls='--', label='OISST clim. std dev')

    for d in month_bounds.values():
        ax.axvline(d, color='gray', lw=0.8, linestyle='--', alpha=0.5)
    for mname, xpos in month_centers:
        ax.text(xpos, ylim[1] * 0.97, mname, ha='center', va='top', color='gray', fontsize=11)

    ax.set_xlabel('Days since beginning of simulation')
    ax.set_ylabel('SST RMSE (°C)')
    ax.set_title(f'SST RMSE vs OISST  |  180–260°E, 10°S–10°N  |  Start: {start_date.strftime("%b %d %Y")}')
    ax.set_xlim(days[0], days[-1])
    ax.set_ylim(ylim)
    ax.grid(alpha=0.3)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=3, borderaxespad=0)

    # ── IQR summary column ────────────────────────────────────────
    ts_data   = [ds[0] for ds in datasets]
    col_names = [f'ds{k}' for k in range(len(ts_data))]
    box_df    = pd.DataFrame({n: pd.Series(d) for n, d in zip(col_names, ts_data)})
    pal_dict  = dict(zip(col_names, palette))
    sns.boxplot(data=box_df, ax=ax_b, palette=pal_dict, width=0.6)
    for k, d in enumerate(ts_data):
        ax_b.plot(k, np.nanmean(d), marker='D', ms=5, color='white', mec='black', mew=0.8, zorder=5)
    ax_b.set_xticks(range(len(short_names)))
    ax_b.set_xticklabels(short_names, fontsize=12, rotation=45, ha='right')
    ax_b.set_ylabel('')
    ax_b.set_ylim(ylim)
    ax_b.grid(alpha=0.3, axis='y')

    plt.savefig(foldername + f'sst_rmse_vs_oisst_{month_str}{day_str}{year_str}{suffix}.png',
                dpi=150, bbox_inches='tight')
    plt.show()